In [2]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import shapely
import pathlib 
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"
NOTEBOOK_PATH

G = ox.load_graphml(
    "turku_bike_network_floodinfo.graphml",
    edge_dtypes={
        "flooded": lambda x: str(x).lower() == "true"
    }
)
flood_area = gpd.read_file(DATA_DIRECTORY/"tulva_kohtalainen"/"Tulva_kohtalainen.shp")

# Remove isolated nodes
G.remove_nodes_from(list(nx.isolates(G)))

# Project graph to EPSG:3067
G_proj = ox.project_graph(G, to_crs="EPSG:3067")

# Prepare individual flood polygons in same CRS
flood = flood_area.to_crs("EPSG:3067").copy()

flood = flood[
    flood.geometry.notna() & ~flood.geometry.is_empty
].copy()

flood = flood[
    flood.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

# Explode multipolygons for spatial indexing
flood = flood.explode(ignore_index=True)

print("Graph CRS:", G_proj.graph["crs"])
print("Original edges:", G_proj.number_of_edges())
print("Flood polygons:", len(flood))

Graph CRS: EPSG:3067
Original edges: 58752
Flood polygons: 269738


In [3]:

from shapely.geometry import LineString
from shapely.ops import substring
import numpy as np

# Minimum length of a sub-edge, in metres.
# Helps avoid nearly zero-length segments from numerical noise.
MIN_SEGMENT = 0.05


def line_parts(geom):
    """Extract non-empty line components from any geometry."""
    if geom.is_empty:
        return []

    if geom.geom_type == "LineString":
        return [geom] if geom.length > 0 else []

    if hasattr(geom, "geoms"):
        parts = []
        for g in geom.geoms:
            parts.extend(line_parts(g))
        return parts

    return []


def split_edge_by_flood(line, flood_geom):
    """
    Return ordered (LineString, flooded) segments.

    line: directed edge geometry in EPSG:3067.
    flood_geom: union of nearby flood polygons.
    """

    if flood_geom is None or not line.intersects(flood_geom):
        return [(line, False)]

    # Find portions of line lying inside the flood extent.
    wet_parts = line_parts(line.intersection(flood_geom))

    if not wet_parts:
        # Point-only contact does not split the line.
        return [(line, False)]

    distances = [0.0, line.length]

    # The ends of wet line portions define the boundaries.
    for part in wet_parts:
        for point in [part.coords[0], part.coords[-1]]:
            distances.append(
                line.project(shapely.Point(point))
            )

    distances.sort()

    # Remove near-duplicate boundary positions
    cuts = []
    for d in distances:
        if not cuts or d - cuts[-1] > MIN_SEGMENT:
            cuts.append(d)

    # Always retain the exact endpoint
    if line.length - cuts[-1] > MIN_SEGMENT:
        cuts.append(line.length)
    else:
        cuts[-1] = line.length

    result = []

    for a, b in zip(cuts[:-1], cuts[1:]):
        segment = substring(line, a, b)

        if segment.length <= 0:
            continue

        # A segment is flooded if a positive length
        # overlaps the flood polygons.
        wet_length = segment.intersection(flood_geom).length
        is_flooded = wet_length > 1e-7

        result.append((segment, is_flooded))

    return result


In [4]:

from collections import Counter

# Work from a copy of the projected OSMnx graph
G_split = G_proj.copy()

# Spatial index of individual flood polygons
flood_sindex = flood.sindex

# Coordinates of newly created boundary nodes
boundary_nodes = {}

# Negative IDs avoid collisions with original OSM IDs
next_node_id = min(0, min(G_split.nodes)) - 1

stats = Counter()


def get_boundary_node(point):
    """
    Create or reuse a node at a flood boundary.

    Rounded coordinates allow reverse-direction edges
    to share the same inserted node.
    """
    global next_node_id

    x, y = point.coords[0]

    # 1 cm coordinate precision
    coordinate_key = (round(x, 2), round(y, 2))

    if coordinate_key in boundary_nodes:
        return boundary_nodes[coordinate_key]

    node_id = next_node_id
    next_node_id -= 1

    G_split.add_node(
        node_id,
        x=float(x),
        y=float(y),
        flood_boundary=True
    )

    boundary_nodes[coordinate_key] = node_id
    return node_id


# Iterate over a snapshot because we modify G_split
for u, v, key, data in list(
    G_proj.edges(keys=True, data=True)
):

    line = data.get("geometry")

    if line is None:
        # Straight geometry for edges with no geometry attribute
        line = LineString([
            (G_proj.nodes[u]["x"], G_proj.nodes[u]["y"]),
            (G_proj.nodes[v]["x"], G_proj.nodes[v]["y"])
        ])

    if line.geom_type != "LineString":
        raise ValueError(
            f"Unexpected geometry for edge {(u, v, key)}: "
            f"{line.geom_type}"
        )

    # Ensure geometry runs from u to v
    start = shapely.Point(
        G_proj.nodes[u]["x"],
        G_proj.nodes[u]["y"]
    )

    if (
        shapely.Point(line.coords[-1]).distance(start)
        < shapely.Point(line.coords[0]).distance(start)
    ):
        line = LineString(list(line.coords)[::-1])

    # Query only polygons potentially intersecting this edge
    candidate_idx = flood_sindex.query(
        line, predicate="intersects"
    )

    if len(candidate_idx) == 0:
        # No flood exposure: keep original edge
        G_split[u][v][key]["flooded"] = False
        continue

    nearby_flood = flood.geometry.iloc[
        candidate_idx
    ].union_all()

    segments = split_edge_by_flood(
        line, nearby_flood
    )

    # No actual line overlap with the flood area
    if len(segments) == 1:
        G_split[u][v][key]["flooded"] = segments[0][1]
        continue

    # Replace original edge with the split sections
    G_split.remove_edge(u, v, key)

    current_node = u

    original_length = float(data["length"])
    geometric_length = line.length

    for i, (segment, is_flooded) in enumerate(segments):

        # Last segment ends at original destination
        if i == len(segments) - 1:
            next_node = v
        else:
            end_point = shapely.Point(segment.coords[-1])
            next_node = get_boundary_node(end_point)

        attributes = data.copy()

        attributes["geometry"] = segment
        attributes["flooded"] = bool(is_flooded)

        # Distribute original routing length proportionally
        attributes["length"] = (
            original_length
            * segment.length / geometric_length
        )

        attributes["original_u"] = u
        attributes["original_v"] = v
        attributes["original_key"] = key

        # New key assigned automatically by NetworkX
        G_split.add_edge(
            current_node,
            next_node,
            **attributes
        )

        current_node = next_node

        stats["created_segments"] += 1
        stats["wet_segments"] += int(is_flooded)

    stats["split_original_edges"] += 1


print("Original nodes:", G_proj.number_of_nodes())
print("Original edges:", G_proj.number_of_edges())

print("Split nodes:", G_split.number_of_nodes())
print("Split edges:", G_split.number_of_edges())

print("Edges replaced:", stats["split_original_edges"])
print("Flooded sub-edges:", stats["wet_segments"])


Original nodes: 24381
Original edges: 58752
Split nodes: 53761
Split edges: 115385
Edges replaced: 15973
Flooded sub-edges: 33054


In [5]:
def total_length(G):
    return sum(
        float(data["length"])
        for _, _, data in G.edges(data=True)
    )

original_length = total_length(G_proj)
split_length = total_length(G_split)

print(f"Original: {original_length / 1000:,.2f} km")
print(f"Split:    {split_length / 1000:,.2f} km")

print(
    "Difference:",
    round(split_length - original_length, 3),
    "m"
)

Original: 3,380.91 km
Split:    3,380.91 km
Difference: -0.0 m


In [6]:
def graph_stats(G, name):
    components = sorted(
        nx.weakly_connected_components(G),
        key=len,
        reverse=True
    )

    print(f"\n{name}")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())
    print("Weak components:", len(components))
    print("Largest component:", len(components[0]))
    print(
        "Largest component (%):",
        round(
            len(components[0]) / G.number_of_nodes() * 100,
            1
        )
    )


graph_stats(G_proj, "Original network")
graph_stats(G_split, "Split network")


Original network
Nodes: 24381
Edges: 58752
Weak components: 424
Largest component: 23309
Largest component (%): 95.6

Split network
Nodes: 53761
Edges: 115385
Weak components: 424
Largest component: 52124
Largest component (%): 97.0


In [7]:
G_split_flooded = G_split.copy()

flooded_edges_to_remove = [
    (u, v, key)
    for u, v, key, data in G_split_flooded.edges(
        keys=True, data=True
    )
    if data.get("flooded", False)
]

G_split_flooded.remove_edges_from(
    flooded_edges_to_remove
)

# Remove nodes with no remaining connections
G_split_flooded.remove_nodes_from(
    list(nx.isolates(G_split_flooded))
)

graph_stats(G_split_flooded, "Split flooded network")


Split flooded network
Nodes: 50190
Edges: 78631
Weak components: 11768
Largest component: 4376
Largest component (%): 8.7


In [10]:
# Save the complete split graph
ox.save_graphml(
    G_split,
    filepath="turku_bike_split.graphml"
)

# Save the split graph with flooded edges removed
ox.save_graphml(
    G_split_flooded,
    filepath="turku_bike_split_flooded.graphml"
)

# Save geospatial layers for visualization
ox.save_graph_geopackage(
    G_split,
    filepath="turku_bike_split.gpkg",
    directed=True
)